In [1]:
# loading in the documentw

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
len(documents)

72

In [3]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

### Generating Ground Truth

In [4]:
# Generating the questions
# The following are the instructions for the LLM to generate 5 questions for each lesson page.

data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()


In [5]:
# We want the output questions as a list of strings, 
# so we define that structure with a Pydantic model.

from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
from evaluation_utils import llm_structured_retry
import json

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
# Test generating ground truth using first 3 documents

from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [9]:
ground_truth

[{'question': 'What is retrieval-augmented generation actually doing to make an LLM answer better?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why do people say LLMs are like a black box in this course?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What kinds of problems do LLMs have that RAG is meant to fix?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What is the course building in this module as a real example of RAG?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What will be covered in the first part of the module versus the agentic second part?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What do I need installed before I can follow this module, and is Jupyter enough by itself?',
  'document': '01-agentic-rag/lessons/02-environment.md'},
 {'question': 'How do I set up a fresh project for this course from scratch with uv?',
  'document': '01-agentic-rag/lesson

In [10]:
usages

[ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=100, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1120),
 ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=114, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1400),
 ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=98, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1851)]

In [11]:
# Question 1: What is the average number of input tokens across these 3 calls?

# input tokens from call 1 = 1020
# input tokens from call 2 = 1286
# input tokens from call 1 = 1753

average = (1020 + 1286 + 1753)/3
average

1353.0

### Getting the full ground truth

In [12]:
# Make a list of records from the downloaded prepared full set of ground truth

import pandas as pd

df_ground_truth = pd.read_csv("data/ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [13]:
len(ground_truth)

360

In [14]:
type(ground_truth)

list

In [15]:
ground_truth[:10]

[{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why does this course build the RAG project in plain Python instead of starting with a framework or library?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are the main weaknesses of large language models that this module is trying to work around?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What will the course build in the first part of the module, and how is the second part different?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What kind of example app are you building here, and what data will it answer questions from?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What do I need installed before starting this module?',
  'filename': '01-agentic-rag/lessons/02-environmen

### Searching the chunks

In [16]:
# Chunk the documents

from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [17]:
len(chunks)

295

In [18]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [19]:
# Build a text index (Index) and a vector index (VectorSearch), both keyed on 'filename'.

from minsearch import Index

def text_search(query, num_results=5):
    
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
    index.fit(chunks)

    search_results = index.search(
        query,
        num_results=num_results
    )

    return search_results

In [20]:
# Create the embeddings in batches and store in X

from embedder import Embedder
from tqdm.auto import tqdm
import numpy as np

embed = Embedder()

texts = [ chunk['content'] for chunk in chunks ]

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

2026-07-16 07:37:47.572916426 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


  0%|          | 0/6 [00:00<?, ?it/s]

In [21]:
from tqdm.auto import tqdm
import numpy as np
from minsearch import VectorSearch


def vector_search(query, num_results=5):
   
    vindex = VectorSearch(keyword_fields=["filename"])
    vindex.fit(X, chunks)

    v_query = embed.encode(query)
    search_results = vindex.search(v_query, num_results=num_results)

    return search_results

In [22]:
# Take the first question from the ground truth

q = ground_truth[0]["question"]
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [23]:
# Run text_search on the first question

text_search_results = text_search(q, num_results=5)

In [24]:
text_search_results

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [25]:
# Run vector_search on the first question

vector_search_results = vector_search(q, num_results=5)

In [26]:
vector_search_results

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [27]:
# Question 2: After running text_search for first question, what is the filename of the first result?
# Answer: 01-agentic-rag/lessons/03-rag.md

# Question 3: After running vector_search on the same question, what is the filename of the first result?
# Answer: 01-agentic-rag/lessons/01-intro.md

### Evaluation Metrics

In [28]:
ground_truth[0]

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

In [35]:
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [36]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [37]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [38]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [39]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [40]:
# Evaluate text_search on the ground truth data

text_search_eval = evaluate( ground_truth, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [41]:
text_search_eval

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [42]:
# Question 4: Evaluate text_search. What is the hit rate?

# Answer: 'hit_rate': 0.7583333333333333 or approx. 0.76

In [43]:
# Evaluate vector_search on the ground truth data

vector_search_eval = evaluate(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [44]:
vector_search_eval

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

In [45]:
# Question 5: Evaluate vector_search.  What is the MRR?
# Answer: 'mrr': 0.5486111111111112 or approx 0.55

### Tuning Hybrid Search

In [46]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [47]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [50]:
# Evaluate hybrid_search over full ground_truth for k values 1, 50, 100, 200

k_values = [1, 50, 100, 200]

results = []

for k in k_values:
    res = {}
    res["k"] = k
    res["eval"] = evaluate( ground_truth, lambda query, k=k: hybrid_search(query, k) )
    results.append(res)

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

In [51]:
results

[{'k': 1, 'eval': {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}},
 {'k': 50, 'eval': {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}},
 {'k': 100,
  'eval': {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}},
 {'k': 200,
  'eval': {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}}]

In [ ]:
# Question 6: Which k gives the best MRR?
# Answer: k=1 gives best mrr = 0.65